# Phase 5: deference direction on Qwen2.5-7B-Instruct (bf16) + scoring-noise check

**Same GPU as before (A100)**, then `Runtime → Run all`. Needs the Phase 4/4b outputs on Drive.

1. GPU check + **smoke test** of every new script on 2 questions (throwaway folder).
2. **Noise check**: Phase 4 vs 4b scored the same prompts in different batches and one prompt moved by 0.93 in P(correct).
   Re-scores the most-changed prompts one at a time and in length-sorted batches, in bf16 and then in fp32 (reference).
3. **Activations + vectors + sanity checks**: last-token residual stream for all ~42k prompts (≈8.7 GB, kept on the
   Colab disk, not Drive), senior − junior vectors (pooled / per template / family / pair / shuffled), and the 4 SPEC sanity
   checks + leave-one-pair-out. Small outputs (`vectors.pt`, `projections.pt`, `sanity_*.csv`) go to Drive.
4. Summary to paste back, then auto-disconnect. Every step stops `Run all` if it fails.

In [ ]:
CFG = "configs/colab.yaml"   # same config as Phase 4
AUTO_DISCONNECT = True
SAVE_ACTS_TO_DRIVE = False    # True copies the ~8.7 GB activations to Drive (needs the space)

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None
url = f"https://{token}@github.com/GTRAN7/vnpronouns.git" if token else "https://github.com/GTRAN7/vnpronouns.git"
if not os.path.exists('/content/vnpronouns'):
    subprocess.run(['git', 'clone', '-q', url, '/content/vnpronouns'], check=True)
else:
    subprocess.run(['git', '-C', '/content/vnpronouns', 'pull', '-q'], check=True)
%cd /content/vnpronouns
!git log --oneline -1
!pip install -q -e .

def run(cmd, log, env=None):
    """Run a shell command, stream + log its output, and raise if it fails (so Run all stops).
    Splits only on real newlines and keeps the text after the last carriage return, so tqdm
    progress bars collapse to one line instead of flooding the notebook."""
    print(f'$ {cmd}', flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1', **(env or {})})
    with open(log, 'a', encoding='utf-8') as f:
        for raw in p.stdout:
            line = raw.decode('utf-8', 'replace').rstrip('\n').split('\r')[-1]
            if line.strip():
                print(line, flush=True)
                f.write(line + '\n')
    if p.wait() != 0:
        raise RuntimeError(f'FAILED (exit {p.returncode}): {cmd}  -- see {log}')

## 1. GPU check + smoke test

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> A100 (or L4)'
name = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'{name}, {mem:.0f} GB, bf16 supported: {torch.cuda.is_bf16_supported()}')
assert torch.cuda.is_bf16_supported() and mem >= 22, f'{name} cannot hold the 7B in bf16; switch to A100/L4'

import os, shutil, time
import pandas as pd
OUT = '/content/drive/MyDrive/vnpronouns/outputs/Qwen2.5-7B-Instruct'
assert os.path.exists(f'{OUT}/behavior_v1_3templates.csv'), 'Phase 4 backup missing: run colab_7b_templates.ipynb first'
assert pd.read_csv(f'{OUT}/behavior.csv', usecols=['template']).template.nunique() == 9, 'behavior.csv should have 9 templates'

SMOKE = {'VNP_OUTPUT_DIR': '/content/smoke'}
SD = '/content/smoke/Qwen2.5-7B-Instruct'
shutil.rmtree('/content/smoke', ignore_errors=True)
t0 = time.time()
run(f'python -m vnpronouns.run_behavior --config {CFG} --pool 2 --no-filter', '/content/log_smoke.txt', SMOKE)
b = pd.read_csv(f'{SD}/behavior.csv')
b[b.template.isin(['flat', 'certain', 'soft'])].to_csv(f'{SD}/behavior_v1_3templates.csv', index=False)
run(f'python -m vnpronouns.noise_check --config {CFG} --n-top 4 --n-random 4', '/content/log_smoke.txt', SMOKE)
run(f'python -m vnpronouns.extract --config {CFG} --acts-path /content/smoke/acts.pt', '/content/log_smoke.txt', SMOKE)
print(f'\nSMOKE TEST PASSED in {time.time() - t0:.0f}s')
shutil.rmtree('/content/smoke', ignore_errors=True)

## 2. Noise check: bf16, then fp32 reference

In [ ]:
!rm -f /content/log_noise.txt
run(f'python -m vnpronouns.noise_check --config {CFG}', '/content/log_noise.txt')
run(f'python -m vnpronouns.noise_check --config {CFG}', '/content/log_noise.txt', {'VNP_DTYPE': 'float32'})

## 3. Activations, vectors, sanity checks

In [ ]:
!rm -f /content/log_extract.txt
run(f'python -m vnpronouns.extract --config {CFG} --acts-path /content/acts/activations.pt', '/content/log_extract.txt')
if SAVE_ACTS_TO_DRIVE:
    shutil.copy('/content/acts/activations.pt', f'{OUT}/activations.pt')

## 4. Summary to paste back to Claude

In [ ]:
import re, torch
def clean(path):
    lines = open(path, encoding='utf-8').read().splitlines()
    return '\n'.join(l.rstrip() for l in lines
                     if not re.search(r'it/s\]|s/it\]|Warning|warn\(|^\s*$|Loading|Fetching|Downloading|%\||^  chunk |^\$ ', l))
print(f'GPU: {torch.cuda.get_device_name(0)}  config: {CFG}')
for p in ['/content/log_noise.txt', '/content/log_extract.txt']:
    print(f'===== {p.split("/")[-1]} =====')
    print(clean(p))
!ls -la /content/drive/MyDrive/vnpronouns/outputs/Qwen2.5-7B-Instruct/ /content/acts/

In [ ]:
if AUTO_DISCONNECT:
    from google.colab import runtime
    print('Done. Disconnecting the runtime to save compute units (outputs above stay visible).')
    runtime.unassign()